# EP1 FINAL



In [ ]:
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from google.colab import drive
from tqdm import tqdm

# ==========================================
# 0. AMBIENTE E SEED OFICIAL (123)
# ==========================================

SEED = 123
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Utilizando dispositivo: {device}")

DATA_PATH = "/content/drive/MyDrive/epPLN/train.xlsx"
df = pd.read_excel(DATA_PATH)

def normalizar_texto(texto):
    if not isinstance(texto, str):
        return ""
    texto = re.sub(r'https?://\S+|www\.\S+', 'LINK_URL', texto)
    texto = re.sub(r'(?i)\b(lei|decreto)\s*(?:nº|n°|nº\.|º|°)?\s*\d+[\d\./]*', 'LEI_DECRETO', texto)
    texto = re.sub(r'\b\d{5,}\.\d{6}/\d{4}-\d{2}\b', 'NUMERO_PROCESSO', texto)
    texto = re.sub(r'\(\?\d{2,4}\)?\s*\d{4,5}[-\s]?\d{4}', 'NUMERO_TELEFONE', texto)
    return texto.lower()

df["resp_text_clean"] = df["resp_text"].apply(normalizar_texto)

label2id = {"c1": 0, "c234": 1, "c5": 2}
id2label = {v: k for k, v in label2id.items()}
df["label"] = df["clarity"].map(label2id)

df_train, df_val = train_test_split(df, test_size=0.20, random_state=SEED, stratify=df["label"])

# ==========================================
# 1. MODELO LARGE (335M Parâmetros)
# ==========================================
MODEL_NAME = 'neuralmind/bert-large-portuguese-cased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LEN = 256

def encode_texts(texts):
    return tokenizer([str(t) for t in texts], padding='max_length', truncation=True, max_length=MAX_LEN, return_tensors="pt")

class TextDataset(Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = torch.tensor(labels, dtype=torch.long) if labels is not None else None

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.encodings['input_ids'])

train_dataset = TextDataset(encode_texts(df_train["resp_text_clean"].to_numpy()), df_train["label"].to_numpy())
val_dataset = TextDataset(encode_texts(df_val["resp_text_clean"].to_numpy()), df_val["label"].to_numpy())

# Batch size 8 para caber na VRAM do Colab
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3).to(device)

# ==========================================
# 2. HIPERPARÂMETROS PARA MODELOS LARGE
# ==========================================
epochs = 4
lr = 1e-5 # Learning rate menor para não desestabilizar as 24 camadas
accum_steps = 4 # Batch efetivo = 8 * 4 = 32

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
total_steps = (len(train_loader) // accum_steps) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps*0.1), num_training_steps=total_steps)
criterion = nn.CrossEntropyLoss()

# ==========================================
# 3. LOOP DE TREINO E VALIDAÇÃO
# ==========================================
print("\nIniciando Treinamento com BERTIMBAU LARGE...")
best_acc = 0.0

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()

    for step, batch in enumerate(tqdm(train_loader, desc=f"Época {epoch+1}/{epochs} [Treino]")):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(outputs.logits, labels) / accum_steps
        loss.backward()

        if (step + 1) % accum_steps == 0 or (step + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

    # Validação
    model.eval()
    preds_all, labels_all = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1)
            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(batch['labels'].numpy())

    acc = accuracy_score(labels_all, preds_all)
    f1_macro = f1_score(labels_all, preds_all, average='macro')
    print(f"\n--> Época {epoch+1} | Val Acc: {acc:.4f} | Val F1-Macro: {f1_macro:.4f}")

    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), 'bertimbau_large_best.pt')
        print(f"    [NOVO RECORD] Modelo LARGE salvo! (Acc: {acc:.4f})")

# ==========================================
# 4. AVALIAÇÃO FINAL DO CAMPEÃO LARGE
# ==========================================
print(f"\n==================================================")
print(f"RESULTADO FINAL DO BERTIMBAU LARGE (Pico: {best_acc:.4f})")
print("==================================================")
model.load_state_dict(torch.load('bertimbau_large_best.pt'))
model.eval()

preds_all, labels_all = [], []
with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)
        preds_all.extend(preds.cpu().numpy())
        labels_all.extend(batch['labels'].numpy())

print(classification_report(labels_all, preds_all, target_names=["c1", "c234", "c5"], digits=4))

# EP2

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from google.colab import drive
from tqdm.notebook import tqdm

# 1. Montar o Google Drive
drive.mount("/content/drive")

# ==========================================
# 0. AMBIENTE E DIREITÓRIOS
# ==========================================
SEED = 123
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_DIR = Path("/content/drive/MyDrive/epPLN/ep2")
TRAIN_PATH = BASE_DIR / "train.xlsx"
TEST2_PATH = BASE_DIR / "test2.xlsx"
OUTPUT_TEST2_PATH = BASE_DIR / "grupoXX_teste2_rotulado.xlsx"

label2id = {"c1": 0, "c234": 1, "c5": 2}
id2label = {v: k for k, v in label2id.items()}

def normalizar_texto(texto):
    if not isinstance(texto, str):
        return ""
    texto = re.sub(r"https?://\S+|www\.\S+", "LINK_URL", texto)
    texto = re.sub(
        r"(?i)\b(lei|decreto)\s*(?:nº|n°|nº\.|º|°)?\s*\d+[\d\./]*",
        "LEI_DECRETO",
        texto,
    )
    texto = re.sub(r"\b\d{5,}\.\d{6}/\d{4}-\d{2}\b", "NUMERO_PROCESSO", texto)
    texto = re.sub(
        r"\(\?\d{2,4}\)?\s*\d{4,5}[-\s]?\d{4}", "NUMERO_TELEFONE", texto
    )
    return texto.lower()

# Carregamento e Pré-processamento
df_train_full = pd.read_excel(TRAIN_PATH)
df_train_full["resp_text_clean"] = df_train_full["resp_text"].apply(normalizar_texto)
df_train_full["label"] = df_train_full["clarity"].map(label2id)

df_train, df_val = train_test_split(
    df_train_full,
    test_size=0.15,
    random_state=SEED,
    stratify=df_train_full["label"],
)

# ==========================================
# 1. CLASSE GENÉRICA DE CLASSIFICAÇÃO
# ==========================================
class GenericTransformerClassifier(nn.Module):
    def __init__(self, model_name, num_classes=3, dropout_rate=0.2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(512, num_classes),
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = outputs.last_hidden_state

        # Mean Pooling
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        mean_pooled = sum_embeddings / sum_mask

        return self.classifier(mean_pooled)

class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=256, labels=None):
        self.encodings = tokenizer(
            [str(t) for t in texts],
            padding="max_length",
            truncation=True,
            max_length=max_len,
            return_tensors="pt",
        )
        self.labels = (
            torch.tensor(labels, dtype=torch.long) if labels is not None else None
        )

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.encodings["input_ids"])

# ==========================================
# 2. TREINAMENTO DOS MODELOS DO ENSEMBLE
# ==========================================
MODELS_TO_TRAIN = [
    {"name": "neuralmind/bert-large-portuguese-cased", "lr": 8e-6, "batch_size": 8, "accum": 4},
    {"name": "neuralmind/bert-base-portuguese-cased", "lr": 2e-5, "batch_size": 16, "accum": 2},
]

val_probabilities = {}
test2_probabilities = {}

df_test2 = pd.read_excel(TEST2_PATH)
df_test2["resp_text_clean"] = df_test2["resp_text"].apply(normalizar_texto)

for model_info in MODELS_TO_TRAIN:
    model_name = model_info["name"]
    print(f"\n🚀 Treinando modelo para o Ensemble: {model_name}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    train_loader = DataLoader(
        TextDataset(df_train["resp_text_clean"].to_numpy(), tokenizer, labels=df_train["label"].to_numpy()),
        batch_size=model_info["batch_size"],
        shuffle=True,
    )
    val_loader = DataLoader(
        TextDataset(df_val["resp_text_clean"].to_numpy(), tokenizer, labels=df_val["label"].to_numpy()),
        batch_size=16,
        shuffle=False,
    )
    test2_loader = DataLoader(
        TextDataset(df_test2["resp_text_clean"].to_numpy(), tokenizer),
        batch_size=16,
        shuffle=False,
    )

    model = GenericTransformerClassifier(model_name, num_classes=3).to(device)

    epochs = 3
    optimizer = torch.optim.AdamW(model.parameters(), lr=model_info["lr"], weight_decay=0.01)
    total_steps = (len(train_loader) // model_info["accum"]) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps
    )
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        for step, batch in enumerate(tqdm(train_loader, desc=f"Época {epoch+1}/{epochs}")):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels) / model_info["accum"]
            loss.backward()

            if (step + 1) % model_info["accum"] == 0 or (step + 1) == len(train_loader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

    # Prever probabilidades na validação
    model.eval()
    probs_val = []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            logits = model(input_ids, attention_mask)
            probs = torch.softmax(logits, dim=1)
            probs_val.extend(probs.cpu().numpy())
    val_probabilities[model_name] = np.array(probs_val)

    # Prever probabilidades no Teste 2
    probs_t2 = []
    with torch.no_grad():
        for batch in test2_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            logits = model(input_ids, attention_mask)
            probs = torch.softmax(logits, dim=1)
            probs_t2.extend(probs.cpu().numpy())
    test2_probabilities[model_name] = np.array(probs_t2)

# ==========================================
# 3. COMBINAÇÃO DOS MODELOS (SOFT VOTING ENSEMBLE)
# ==========================================
print("\n==========================================")
print("AVALIANDO O DESEMPENHO DO ENSEMBLE...")
print("==========================================")

y_val_true = df_val["label"].to_numpy()

# Média simples das probabilidades dos modelos do ensemble
ensemble_val_probs = np.mean(list(val_probabilities.values()), axis=0)
ensemble_val_preds = np.argmax(ensemble_val_probs, dim=1) if isinstance(ensemble_val_probs, torch.Tensor) else np.argmax(ensemble_val_probs, axis=1)

acc_ensemble = accuracy_score(y_val_true, ensemble_val_preds)
print(f"🔥 Acurácia Final do Ensemble no Conjunto de Validação: {acc_ensemble:.4f}")


Mounted at /content/drive

🚀 Treinando modelo para o Ensemble: neuralmind/bert-large-portuguese-cased


config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/155 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

... (21 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Época 2/3:   0%|          | 0/1068 [00:00<?, ?it/s]

Época 3/3:   0%|          | 0/1068 [00:00<?, ?it/s]


AVALIANDO O DESEMPENHO DO ENSEMBLE...
🔥 Acurácia Final do Ensemble no Conjunto de Validação: 0.4678


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import accuracy_score
from statsmodels.stats.contingency_tables import mcnemar
from google.colab import drive

# 1. Montar o Google Drive (se necessário)
#drive.mount("/content/drive")

# ==========================================
# 0. AMBIENTE E DIREITÓRIOS
# ==========================================
SEED = 123
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_DIR = Path("/content/drive/MyDrive/epPLN/ep2")
TEST1_PATH = BASE_DIR / "test1-rotulado-para-teste-estatistico.xlsx"
EP1_PREDS_PATH = BASE_DIR / "test1_rotulado.xlsx"

label2id = {"c1": 0, "c234": 1, "c5": 2}
id2label = {v: k for k, v in label2id.items()}

def normalizar_texto(texto):
    if not isinstance(texto, str):
        return ""
    texto = re.sub(r"https?://\S+|www\.\S+", "LINK_URL", texto)
    texto = re.sub(
        r"(?i)\b(lei|decreto)\s*(?:nº|n°|nº\.|º|°)?\s*\d+[\d\./]*",
        "LEI_DECRETO",
        texto,
    )
    texto = re.sub(r"\b\d{5,}\.\d{6}/\d{4}-\d{2}\b", "NUMERO_PROCESSO", texto)
    texto = re.sub(
        r"\(\?\d{2,4}\)?\s*\d{4,5}[-\s]?\d{4}", "NUMERO_TELEFONE", texto
    )
    return texto.lower()

# Carregar o gabarito oficial do test1 e as predições do EP1
df_test1 = pd.read_excel(TEST1_PATH)
df_test1["resp_text_clean"] = df_test1["resp_text"].apply(normalizar_texto)
y_true = df_test1["clarity"].map(label2id).to_numpy()

df_ep1 = pd.read_excel(EP1_PREDS_PATH)
preds_ep1 = df_ep1["clarity"].map(label2id).to_numpy()

# ==========================================
# 1. INFERÊNCIA DO ENSEMBLE NO TEST1
# ==========================================
# (Se os modelos do ensemble já estiverem instanciados na memória do Colab,
# você pode usar o loop abaixo para extrair as probabilidades)

test1_probabilities = {}

for model_info in MODELS_TO_TRAIN:
    model_name = model_info["name"]
    print(f"Obtendo probabilidades no Teste 1 para: {model_name}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    test1_loader = DataLoader(
        TextDataset(df_test1["resp_text_clean"].to_numpy(), tokenizer),
        batch_size=16,
        shuffle=False
    )

    # Supondo que 'model' foi mantido ou recarregado para cada arquitetura
    # (Caso tenha os checkpoints salvos, carregue-os aqui)
    model.eval()
    probs_t1 = []
    with torch.no_grad():
        for batch in test1_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            logits = model(input_ids, attention_mask)
            probs = torch.softmax(logits, dim=1)
            probs_t1.extend(probs.cpu().numpy())

    test1_probabilities[model_name] = np.array(probs_t1)

# Soft Voting: Média das probabilidades no Teste 1
ensemble_test1_probs = np.mean(list(test1_probabilities.values()), axis=0)
preds_ensemble_ep2 = np.argmax(ensemble_test1_probs, axis=1)

# ==========================================
# 2. COMPARATIVO DE ACURÁCIA E MCNEMAR
# ==========================================
acc_ep1 = accuracy_score(y_true, preds_ep1)
acc_ep2_ensemble = accuracy_score(y_true, preds_ensemble_ep2)

print("\n==========================================")
print("AVALIAÇÃO DE ACURÁCIA NO TESTE 1 (900 Instâncias)")
print("==========================================")
print(f"Acurácia EP1 (Modelo Anterior): {acc_ep1:.4f} ({acc_ep1*100:.2f}%)")
print(f"Acurácia EP2 (Ensemble Híbrido): {acc_ep2_ensemble:.4f} ({acc_ep2_ensemble*100:.2f}%)")

# Tabela de Contingência
correct_ep1 = (preds_ep1 == y_true)
correct_ep2 = (preds_ensemble_ep2 == y_true)

n11 = np.sum(correct_ep1 & correct_ep2)   # Ambos acertaram
n10 = np.sum(correct_ep1 & ~correct_ep2)  # EP1 acertou, EP2 errou
n01 = np.sum(~correct_ep1 & correct_ep2)  # EP1 errou, EP2 acertou
n00 = np.sum(~correct_ep1 & ~correct_ep2) # Ambos erraram

table = [[n11, n10],
         [n01, n00]]

print(f"\nTabela de Contingência (McNemar):")
print(f"               EP2 Correto | EP2 Incorreto")
print(f"EP1 Correto   :    {n11:^7} |    {n10:^7}")
print(f"EP1 Incorreto :    {n01:^7} |    {n00:^7}\n")

mcnemar_result = mcnemar(table, exact=False, correction=True)
print(f"Estatística qui-quadrado (χ²): {mcnemar_result.statistic:.4f}")
print(f"p-valor: {mcnemar_result.pvalue:.6f}")

if mcnemar_result.pvalue < 0.05:
    print("\n✅ O Ensemble no EP2 é ESTATISTICAMENTE SUPERIOR ao EP1 (p < 0,05)!")
else:
    print("\n⚠️ A diferença de desempenho entre os modelos não atingiu significância estatística (p >= 0,05).")

Obtendo probabilidades no Teste 1 para: neuralmind/bert-large-portuguese-cased
Obtendo probabilidades no Teste 1 para: neuralmind/bert-base-portuguese-cased

AVALIAÇÃO DE ACURÁCIA NO TESTE 1 (900 Instâncias)
Acurácia EP1 (Modelo Anterior): 0.5022 (50.22%)
Acurácia EP2 (Ensemble Híbrido): 0.4778 (47.78%)

Tabela de Contingência (McNemar):
               EP2 Correto | EP2 Incorreto
EP1 Correto   :      341   |      111  
EP1 Incorreto :      89    |      359  

Estatística qui-quadrado (χ²): 2.2050
p-valor: 0.137564

⚠️ A diferença de desempenho entre os modelos não atingiu significância estatística (p >= 0,05).


In [ ]:
# ==========================================
# 4. ROTULAR O TESTE 2 COM O ENSEMBLE
# ==========================================
ensemble_test2_probs = np.mean(list(test2_probabilities.values()), axis=0)
ensemble_test2_preds = np.argmax(ensemble_test2_probs, axis=1)

df_test2["clarity"] = [id2label[p] for p in ensemble_test2_preds]
df_test2_out = df_test2.drop(columns=["resp_text_clean"])
df_test2_out.to_excel(OUTPUT_TEST2_PATH, index=False)

print(f"\n✅ Planilha Oficial do Teste 2 gerada com sucesso via Ensemble em: {OUTPUT_TEST2_PATH}")
print(f"Total de instâncias rotuladas: {len(df_test2_out)}")

melhoria ep1: Para mitigar a saturação de gradiente e capturar melhor o contexto global de respostas mais longas, substituiu-se a extração tradicional baseada apenas no vetor do token especial [CLS] por uma estratégia de Mean Pooling (Média Ponderada das Representações).

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from google.colab import drive
from tqdm.notebook import tqdm

# 1. Montar o Google Drive
drive.mount("/content/drive")

# ==========================================
# 0. AMBIENTE E SEED
# ==========================================
SEED = 123
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_DIR = Path("/content/drive/MyDrive/epPLN/ep2")
TRAIN_PATH = BASE_DIR / "train.xlsx"
TEST2_PATH = BASE_DIR / "test2.xlsx"
SAVE_ACC_MODEL_PATH = BASE_DIR / "bertimbau_large_best_acc.pt"
OUTPUT_TEST2_PATH = BASE_DIR / "grupoXX_teste2_rotulado.xlsx"

label2id = {"c1": 0, "c234": 1, "c5": 2}
id2label = {v: k for k, v in label2id.items()}

def normalizar_texto(texto):
    if not isinstance(texto, str):
        return ""
    texto = re.sub(r"https?://\S+|www\.\S+", "LINK_URL", texto)
    texto = re.sub(
        r"(?i)\b(lei|decreto)\s*(?:nº|n°|nº\.|º|°)?\s*\d+[\d\./]*",
        "LEI_DECRETO",
        texto,
    )
    texto = re.sub(r"\b\d{5,}\.\d{6}/\d{4}-\d{2}\b", "NUMERO_PROCESSO", texto)
    texto = re.sub(
        r"\(\?\d{2,4}\)?\s*\d{4,5}[-\s]?\d{4}", "NUMERO_TELEFONE", texto
    )
    return texto.lower()

# Carregamento e Pré-processamento
df_train_full = pd.read_excel(TRAIN_PATH)
df_train_full["resp_text_clean"] = df_train_full["resp_text"].apply(normalizar_texto)
df_train_full["label"] = df_train_full["clarity"].map(label2id)

df_train, df_val = train_test_split(
    df_train_full,
    test_size=0.15,
    random_state=SEED,
    stratify=df_train_full["label"],
)

# ==========================================
# 1. ARQUITETURA COM MEAN POOLING
# ==========================================
MODEL_NAME = "neuralmind/bert-large-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LEN = 256

class BERTimbauMeanPoolingClassifier(nn.Module):
    def __init__(self, model_name, num_classes=3, dropout_rate=0.2):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size  # 1024

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(512, num_classes),
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = outputs.last_hidden_state

        # Mean Pooling: Média ponderada de todos os tokens válidos
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        mean_pooled = sum_embeddings / sum_mask

        return self.classifier(mean_pooled)

class TextDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.encodings = tokenizer(
            [str(t) for t in texts],
            padding="max_length",
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt",
        )
        self.labels = (
            torch.tensor(labels, dtype=torch.long) if labels is not None else None
        )

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.encodings["input_ids"])

train_loader = DataLoader(
    TextDataset(df_train["resp_text_clean"].to_numpy(), df_train["label"].to_numpy()),
    batch_size=8,
    shuffle=True,
)
val_loader = DataLoader(
    TextDataset(df_val["resp_text_clean"].to_numpy(), df_val["label"].to_numpy()),
    batch_size=16,
    shuffle=False,
)

# ==========================================
# 2. HIPERPARÂMETROS FOCADOS EM ACURÁCIA
# ==========================================
model = BERTimbauMeanPoolingClassifier(MODEL_NAME, num_classes=3, dropout_rate=0.2).to(device)

lr_base = 8e-6
epochs = 4
accum_steps = 4

optimizer_grouped_parameters = [
    {"params": [p for n, p in model.bert.named_parameters()], "lr": lr_base * 0.8},
    {"params": [p for n, p in model.classifier.named_parameters()], "lr": lr_base},
]

optimizer = torch.optim.AdamW(optimizer_grouped_parameters, weight_decay=0.01)
total_steps = (len(train_loader) // accum_steps) * epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * 0.15),
    num_training_steps=total_steps,
)
criterion = nn.CrossEntropyLoss()

# ==========================================
# 3. TREINAMENTO (MÉTRICA CHAVE = ACURÁCIA)
# ==========================================
best_acc = 0.0

print("🚀 TREINANDO BERTIMBAU LARGE COM MEAN POOLING (Foco: Acurácia)...")

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()

    for step, batch in enumerate(tqdm(train_loader, desc=f"Época {epoch+1}/{epochs}")):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels) / accum_steps
        loss.backward()

        if (step + 1) % accum_steps == 0 or (step + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

    # Avaliação em Acurácia
    model.eval()
    preds_all, labels_all = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            logits = model(input_ids, attention_mask)
            preds = torch.argmax(logits, dim=1)
            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(batch["labels"].numpy())

    val_acc = accuracy_score(labels_all, preds_all)
    print(f"--> Época {epoch+1} | Val Acurácia: {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), SAVE_ACC_MODEL_PATH)
        print(f"    [NOVO RECORDE DE ACURÁCIA: {val_acc:.4f}] Modelo salvo!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: neuralmind/bert-large-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🚀 TREINANDO BERTIMBAU LARGE COM MEAN POOLING (Foco: Acurácia)...


Época 1/4:   0%|          | 0/2135 [00:00<?, ?it/s]

--> Época 1 | Val Acurácia: 0.4366
    [NOVO RECORDE DE ACURÁCIA: 0.4366] Modelo salvo!


... (2 outputs left out)

Época 3/4:   0%|          | 0/2135 [00:00<?, ?it/s]

--> Época 3 | Val Acurácia: 0.4466


Época 4/4:   0%|          | 0/2135 [00:00<?, ?it/s]

--> Época 4 | Val Acurácia: 0.4516
    [NOVO RECORDE DE ACURÁCIA: 0.4516] Modelo salvo!


In [ ]:
# ==========================================
# 4. GERAR PREDIÇÕES DO TESTE 2 COM O RECORDE
# ==========================================
print("\n==========================================")
print(f"GERANDO 'grupoXX_teste2_rotulado.xlsx' (Modelo com Acc {best_acc:.4f})")
print("==========================================")

model.load_state_dict(torch.load(SAVE_ACC_MODEL_PATH, map_location=device))
model.eval()

df_test2 = pd.read_excel(TEST2_PATH)
df_test2["resp_text_clean"] = df_test2["resp_text"].apply(normalizar_texto)

test2_loader = DataLoader(
    TextDataset(df_test2["resp_text_clean"].to_numpy()),
    batch_size=16,
    shuffle=False,
)

preds_test2 = []
with torch.no_grad():
    for batch in test2_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        logits = model(input_ids, attention_mask)
        preds = torch.argmax(logits, dim=1)
        preds_test2.extend(preds.cpu().numpy())

df_test2["clarity"] = [id2label[p] for p in preds_test2]
df_test2_out = df_test2.drop(columns=["resp_text_clean"])
df_test2_out.to_excel(OUTPUT_TEST2_PATH, index=False)

print(f"✅ Planilha do Teste 2 salva com sucesso em: {OUTPUT_TEST2_PATH}")
print(f"Total de instâncias rotuladas: {len(df_test2_out)}")

MELHORIA EP1: grid search no BERT escolhido + significância
Variou-se a taxa de aprendizado ($1e-5$, $1.5e-5$, $2e-5$), a taxa de dropout ($0.2$, $0.3$) e o weight decay ($0.01$, $0.05$) no otimizador AdamW, avaliando cada combinação no conjunto de validação para selecionar automaticamente o checkpoint de maior F1-Macro.
PÉSSIMO

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from google.colab import drive
from tqdm.notebook import tqdm

# 1. Montar o Google Drive
drive.mount("/content/drive")

# ==========================================
# 0. AMBIENTE E SEED
# ==========================================
SEED = 123
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_DIR = Path("/content/drive/MyDrive/epPLN/ep2")
TRAIN_PATH = BASE_DIR / "train.xlsx"
SAVE_BEST_GRID_PATH = BASE_DIR / "bertimbau_large_grid_best.pt"

label2id = {"c1": 0, "c234": 1, "c5": 2}
id2label = {v: k for k, v in label2id.items()}

def normalizar_texto(texto):
    if not isinstance(texto, str):
        return ""
    texto = re.sub(r"https?://\S+|www\.\S+", "LINK_URL", texto)
    texto = re.sub(
        r"(?i)\b(lei|decreto)\s*(?:nº|n°|nº\.|º|°)?\s*\d+[\d\./]*",
        "LEI_DECRETO",
        texto,
    )
    texto = re.sub(r"\b\d{5,}\.\d{6}/\d{4}-\d{2}\b", "NUMERO_PROCESSO", texto)
    texto = re.sub(
        r"\(\?\d{2,4}\)?\s*\d{4,5}[-\s]?\d{4}", "NUMERO_TELEFONE", texto
    )
    return texto.lower()

df_train_full = pd.read_excel(TRAIN_PATH)
df_train_full["resp_text_clean"] = df_train_full["resp_text"].apply(normalizar_texto)
df_train_full["label"] = df_train_full["clarity"].map(label2id)

df_train, df_val = train_test_split(
    df_train_full,
    test_size=0.15,
    random_state=SEED,
    stratify=df_train_full["label"],
)

# ==========================================
# 1. ESTRUTURA DO MODELO COM DROPOUT CONFIGURÁVEL
# ==========================================
MODEL_NAME = "neuralmind/bert-large-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LEN = 256

class BERTimbauCustomClassifier(nn.Module):
    def __init__(self, model_name, num_classes=3, dropout_rate=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size  # 1024 no Large

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(512, num_classes),
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        return self.classifier(cls_output)

class TextDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.encodings = tokenizer(
            [str(t) for t in texts],
            padding="max_length",
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt",
        )
        self.labels = (
            torch.tensor(labels, dtype=torch.long) if labels is not None else None
        )

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.encodings["input_ids"])

train_loader = DataLoader(
    TextDataset(df_train["resp_text_clean"].to_numpy(), df_train["label"].to_numpy()),
    batch_size=8,
    shuffle=True,
)
val_loader = DataLoader(
    TextDataset(df_val["resp_text_clean"].to_numpy(), df_val["label"].to_numpy()),
    batch_size=16,
    shuffle=False,
)

# ==========================================
# 2. DEFINIÇÃO DA GRADE (GRID SEARCH)
# ==========================================
grid_params = [
    {"lr": 1e-5, "dropout": 0.2, "weight_decay": 0.01},
    {"lr": 1.5e-5, "dropout": 0.3, "weight_decay": 0.01},
    {"lr": 2e-5, "dropout": 0.3, "weight_decay": 0.05},
]

best_score = 0.0
best_config = None

print("🔍 INICIANDO GRID SEARCH DE HIPERPARÂMETROS...")

for config_idx, params in enumerate(grid_params, start=1):
    print(f"\n--- [Teste {config_idx}/{len(grid_params)}] Testando Configuração: {params} ---")

    model = BERTimbauCustomClassifier(
        MODEL_NAME, num_classes=3, dropout_rate=params["dropout"]
    ).to(device)

    optimizer_grouped_parameters = [
        {"params": [p for n, p in model.bert.named_parameters()], "lr": params["lr"] * 0.8},
        {"params": [p for n, p in model.classifier.named_parameters()], "lr": params["lr"]},
    ]
    optimizer = torch.optim.AdamW(optimizer_grouped_parameters, weight_decay=params["weight_decay"])

    epochs_grid = 2  # 2 épocas para testar rapidamente a viabilidade de cada combinação
    accum_steps = 4
    total_steps = (len(train_loader) // accum_steps) * epochs_grid
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * 0.1),
        num_training_steps=total_steps,
    )
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs_grid):
        model.train()
        optimizer.zero_grad()
        for step, batch in enumerate(train_loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels) / accum_steps
            loss.backward()

            if (step + 1) % accum_steps == 0 or (step + 1) == len(train_loader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

    # Avaliação da Configuração
    model.eval()
    preds_all, labels_all = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            logits = model(input_ids, attention_mask)
            preds = torch.argmax(logits, dim=1)
            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(batch["labels"].numpy())

    val_f1 = f1_score(labels_all, preds_all, average="macro")
    print(f"Resultado Configuração {config_idx}: F1-Macro = {val_f1:.4f}")

    if val_f1 > best_score:
        best_score = val_f1
        best_config = params
        torch.save(model.state_dict(), SAVE_BEST_GRID_PATH)

print("\n==========================================")
print(f"🏆 MELHOR CONFIGURAÇÃO ENCONTRADA NO GRID:")
print(best_config)
print(f"Melhor Val F1-Macro: {best_score:.4f}")
print("==========================================")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔍 INICIANDO GRID SEARCH DE HIPERPARÂMETROS...

--- [Teste 1/3] Testando Configuração: {'lr': 1e-05, 'dropout': 0.2, 'weight_decay': 0.01} ---


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: neuralmind/bert-large-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Resultado Configuração 1: F1-Macro = 0.4505

--- [Teste 2/3] Testando Configuração: {'lr': 1.5e-05, 'dropout': 0.3, 'weight_decay': 0.01} ---


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: neuralmind/bert-large-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Resultado Configuração 2: F1-Macro = 0.4466

--- [Teste 3/3] Testando Configuração: {'lr': 2e-05, 'dropout': 0.3, 'weight_decay': 0.05} ---


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: neuralmind/bert-large-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Resultado Configuração 3: F1-Macro = 0.4439

🏆 MELHOR CONFIGURAÇÃO ENCONTRADA NO GRID:
{'lr': 1e-05, 'dropout': 0.2, 'weight_decay': 0.01}
Melhor Val F1-Macro: 0.4505


nOVO GRID

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from google.colab import drive
from tqdm.notebook import tqdm

# 1. Montar o Google Drive
drive.mount("/content/drive")

# ==========================================
# 0. AMBIENTE E DIREITÓRIOS
# ==========================================
SEED = 123
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_DIR = Path("/content/drive/MyDrive/epPLN/ep2")
TRAIN_PATH = BASE_DIR / "train.xlsx"
TEST2_PATH = BASE_DIR / "test2.xlsx"
SAVE_ACC_MODEL_PATH = BASE_DIR / "bertimbau_large_grid_acc.pt"
OUTPUT_TEST2_PATH = BASE_DIR / "grupoXX_teste2_rotulado.xlsx"

label2id = {"c1": 0, "c234": 1, "c5": 2}
id2label = {v: k for k, v in label2id.items()}

def normalizar_texto(texto):
    if not isinstance(texto, str):
        return ""
    texto = re.sub(r"https?://\S+|www\.\S+", "LINK_URL", texto)
    texto = re.sub(
        r"(?i)\b(lei|decreto)\s*(?:nº|n°|nº\.|º|°)?\s*\d+[\d\./]*",
        "LEI_DECRETO",
        texto,
    )
    texto = re.sub(r"\b\d{5,}\.\d{6}/\d{4}-\d{2}\b", "NUMERO_PROCESSO", texto)
    texto = re.sub(
        r"\(\?\d{2,4}\)?\s*\d{4,5}[-\s]?\d{4}", "NUMERO_TELEFONE", texto
    )
    return texto.lower()

# Carregamento do dataset de treino
df_train_full = pd.read_excel(TRAIN_PATH)
df_train_full["resp_text_clean"] = df_train_full["resp_text"].apply(normalizar_texto)
df_train_full["label"] = df_train_full["clarity"].map(label2id)

df_train, df_val = train_test_split(
    df_train_full,
    test_size=0.15,
    random_state=SEED,
    stratify=df_train_full["label"],
)

# ==========================================
# 1. ARQUITETURA COM MEAN POOLING
# ==========================================
MODEL_NAME = "neuralmind/bert-large-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LEN = 256

class BERTimbauMeanPoolingClassifier(nn.Module):
    def __init__(self, model_name, num_classes=3, dropout_rate=0.2):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size  # 1024

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(512, num_classes),
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = outputs.last_hidden_state

        input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        mean_pooled = sum_embeddings / sum_mask

        return self.classifier(mean_pooled)

class TextDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.encodings = tokenizer(
            [str(t) for t in texts],
            padding="max_length",
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt",
        )
        self.labels = (
            torch.tensor(labels, dtype=torch.long) if labels is not None else None
        )

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.encodings["input_ids"])

train_loader = DataLoader(
    TextDataset(df_train["resp_text_clean"].to_numpy(), df_train["label"].to_numpy()),
    batch_size=8,
    shuffle=True,
)
val_loader = DataLoader(
    TextDataset(df_val["resp_text_clean"].to_numpy(), df_val["label"].to_numpy()),
    batch_size=16,
    shuffle=False,
)

# ==========================================
# 2. DEFINIÇÃO DA NOVA GRID (FOCO EM ESTABILIDADE E ACC)
# ==========================================
grid_params = [
    {"lr": 5e-6, "dropout": 0.2, "weight_decay": 0.01, "warmup": 0.15},
    {"lr": 8e-6, "dropout": 0.2, "weight_decay": 0.01, "warmup": 0.15},
    {"lr": 1e-5, "dropout": 0.3, "weight_decay": 0.05, "warmup": 0.10},
]

best_acc = 0.0
best_config = None

print("🔍 INICIANDO GRID SEARCH OTIMIZADA PARA ACURÁCIA...")

for config_idx, params in enumerate(grid_params, start=1):
    print(f"\n--- [Teste {config_idx}/{len(grid_params)}] Testando Configuração: {params} ---")

    model = BERTimbauMeanPoolingClassifier(
        MODEL_NAME, num_classes=3, dropout_rate=params["dropout"]
    ).to(device)

    optimizer_grouped_parameters = [
        {"params": [p for n, p in model.bert.named_parameters()], "lr": params["lr"] * 0.8},
        {"params": [p for n, p in model.classifier.named_parameters()], "lr": params["lr"]},
    ]
    optimizer = torch.optim.AdamW(optimizer_grouped_parameters, weight_decay=params["weight_decay"])

    epochs_grid = 3  # 3 épocas por teste para avaliar a convergência real
    accum_steps = 4
    total_steps = (len(train_loader) // accum_steps) * epochs_grid
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * params["warmup"]),
        num_training_steps=total_steps,
    )
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs_grid):
        model.train()
        optimizer.zero_grad()
        for step, batch in enumerate(tqdm(train_loader, desc=f"Época {epoch+1}/{epochs_grid}")):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels) / accum_steps
            loss.backward()

            if (step + 1) % accum_steps == 0 or (step + 1) == len(train_loader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

    # Avaliação por Acurácia
    model.eval()
    preds_all, labels_all = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            logits = model(input_ids, attention_mask)
            preds = torch.argmax(logits, dim=1)
            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(batch["labels"].numpy())

    val_acc = accuracy_score(labels_all, preds_all)
    print(f"Result Configuração {config_idx}: Val Acurácia = {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        best_config = params
        torch.save(model.state_dict(), SAVE_ACC_MODEL_PATH)
        print(f"   🏆 [NOVA MELHOR CONFIG] Salva com Val Acc: {val_acc:.4f}")

print("\n==========================================")
print(f"🏆 MELHOR CONFIGURAÇÃO ENCONTRADA:")
print(best_config)
print(f"Pico de Val Acurácia: {best_acc:.4f}")
print("==========================================")

Mounted at /content/drive


config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/155 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

... (11 outputs left out)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: neuralmind/bert-large-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Época 1/3:   0%|          | 0/2135 [00:00<?, ?it/s]

Época 2/3:   0%|          | 0/2135 [00:00<?, ?it/s]

In [ ]:
# ==========================================
# 3. GERAR AS PREDIÇÕES OFICIAIS DO TESTE 2
# ==========================================
print("\nGerando 'grupoXX_teste2_rotulado.xlsx' com a melhor configuração...")

model.load_state_dict(torch.load(SAVE_ACC_MODEL_PATH, map_location=device))
model.eval()

df_test2 = pd.read_excel(TEST2_PATH)
df_test2["resp_text_clean"] = df_test2["resp_text"].apply(normalizar_texto)

test2_loader = DataLoader(
    TextDataset(df_test2["resp_text_clean"].to_numpy()),
    batch_size=16,
    shuffle=False,
)

preds_test2 = []
with torch.no_grad():
    for batch in test2_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        logits = model(input_ids, attention_mask)
        preds = torch.argmax(logits, dim=1)
        preds_test2.extend(preds.cpu().numpy())

df_test2["clarity"] = [id2label[p] for p in preds_test2]
df_test2_out = df_test2.drop(columns=["resp_text_clean"])
df_test2_out.to_excel(OUTPUT_TEST2_PATH, index=False)

print(f"✅ Arquivo final do Teste 2 gerado com sucesso em: {OUTPUT_TEST2_PATH}")

MELHORIA EP1: ajuste manual. otimização de hiperparâmetros ajustando manualmente a learning rate para $1.5e-5$, com a adição de uma cabeça de classificação MLP personalizada (camada linear intermediária de 512 unidades, BatchNorm e Dropout de 0,3 para reduzir overfitting).
## Não é significante

In [ ]:
import os
from pathlib import Path
import re
import numpy as np
import pandas as pd
from scipy.stats import chisquare
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from statsmodels.stats.contingency_tables import mcnemar
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from tqdm.notebook import tqdm
from transformers import (
    AutoModel,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

# ==========================================
# 0. DIREITÓRIOS E SEED (123)
# ==========================================
SEED = 123
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_DIR = Path("/content/drive/MyDrive/epPLN/ep2")
TRAIN_PATH = BASE_DIR / "train.xlsx"
TEST1_PATH = BASE_DIR / "test1-rotulado-para-teste-estatistico.xlsx"
TEST2_PATH = BASE_DIR / "test2.xlsx"
SAVE_MODEL_PATH = BASE_DIR / "bertimbau_large_ep2_best.pt"

label2id = {"c1": 0, "c234": 1, "c5": 2}
id2label = {v: k for k, v in label2id.items()}


# Normalização do texto
def normalizar_texto(texto):
    if not isinstance(texto, str):
        return ""
    texto = re.sub(r"https?://\S+|www\.\S+", "LINK_URL", texto)
    texto = re.sub(
        r"(?i)\b(lei|decreto)\s*(?:nº|n°|nº\.|º|°)?\s*\d+[\d\./]*",
        "LEI_DECRETO",
        texto,
    )
    texto = re.sub(r"\b\d{5,}\.\d{6}/\d{4}-\d{2}\b", "NUMERO_PROCESSO", texto)
    texto = re.sub(
        r"\(\?\d{2,4}\)?\s*\d{4,5}[-\s]?\d{4}", "NUMERO_TELEFONE", texto
    )
    return texto.lower()


# Carregamento do treino
df_train_full = pd.read_excel(TRAIN_PATH)
df_train_full["resp_text_clean"] = df_train_full["resp_text"].apply(
    normalizar_texto
)
df_train_full["label"] = df_train_full["clarity"].map(label2id)

df_train, df_val = train_test_split(
    df_train_full,
    test_size=0.15,
    random_state=SEED,
    stratify=df_train_full["label"],
)

# ==========================================
# 1. ARQUITETURA CUSTOMIZADA COM MLP HEAD
# ==========================================
MODEL_NAME = "neuralmind/bert-large-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LEN = 256


class BERTimbauCustomClassifier(nn.Module):

    def __init__(self, model_name, num_classes=3, dropout_rate=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size  # 1024 para o Large

        # Cabeça MLP avançada para evitar overfitting e melhorar separabilidade
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(512, num_classes),
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # Token [CLS] representa o contexto global
        cls_output = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(cls_output)
        return logits


class TextDataset(Dataset):

    def __init__(self, texts, labels=None):
        self.encodings = tokenizer(
            [str(t) for t in texts],
            padding="max_length",
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt",
        )
        self.labels = (
            torch.tensor(labels, dtype=torch.long) if labels is not None else None
        )

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.encodings["input_ids"])


train_loader = DataLoader(
    TextDataset(
        df_train["resp_text_clean"].to_numpy(), df_train["label"].to_numpy()
    ),
    batch_size=8,
    shuffle=True,
)
val_loader = DataLoader(
    TextDataset(
        df_val["resp_text_clean"].to_numpy(), df_val["label"].to_numpy()
    ),
    batch_size=16,
    shuffle=False,
)

model = BERTimbauCustomClassifier(MODEL_NAME, num_classes=3).to(device)

# ==========================================
# 2. HIPERPARÂMETROS & OTIMIZADOR
# ==========================================
epochs = 4
accum_steps = 4
base_lr = 1.5e-5

# Layer-wise Learning Rate Decay (LLRD)
optimizer_grouped_parameters = [
    {
        "params": [p for n, p in model.bert.named_parameters()],
        "lr": base_lr * 0.8,
    },
    {"params": [p for n, p in model.classifier.named_parameters()], "lr": base_lr},
]

optimizer = torch.optim.AdamW(
    optimizer_grouped_parameters, weight_decay=0.01
)
total_steps = (len(train_loader) // accum_steps) * epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * 0.1),
    num_training_steps=total_steps,
)
criterion = nn.CrossEntropyLoss()

# ==========================================
# 3. TREINAMENTO COM AVALIAÇÃO
# ==========================================
print("🚀 Iniciando Fine-Tuning Avançado do BERTimbau Large...")
best_f1 = 0.0

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()

    for step, batch in enumerate(
        tqdm(train_loader, desc=f"Época {epoch+1}/{epochs}")
    ):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels) / accum_steps
        loss.backward()

        if (step + 1) % accum_steps == 0 or (step + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

    # Validação
    model.eval()
    preds_all, labels_all = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            logits = model(input_ids, attention_mask)
            preds = torch.argmax(logits, dim=1)
            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(batch["labels"].numpy())

    val_f1 = f1_score(labels_all, preds_all, average="macro")
    val_acc = accuracy_score(labels_all, preds_all)
    print(f"--> Época {epoch+1} | Acc: {val_acc:.4f} | F1-Macro: {val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), SAVE_MODEL_PATH)
        print(f"    [RECORD] Melhores pesos salvos em {SAVE_MODEL_PATH}")

# ==========================================
# 4. PREDIÇÃO OFICIAL NO TESTE 2 (897 Instâncias)
# ==========================================
print("\n==========================================")
print("GERANDO PREDIÇÕES OFICIAIS DO TESTE 2")
print("==========================================")

model.load_state_dict(torch.load(SAVE_MODEL_PATH, map_location=device))
model.eval()

df_test2 = pd.read_excel(TEST2_PATH)
df_test2["resp_text_clean"] = df_test2["resp_text"].apply(normalizar_texto)

test2_loader = DataLoader(
    TextDataset(df_test2["resp_text_clean"].to_numpy()),
    batch_size=16,
    shuffle=False,
)

preds_test2 = []
with torch.no_grad():
    for batch in test2_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        logits = model(input_ids, attention_mask)
        preds = torch.argmax(logits, dim=1)
        preds_test2.extend(preds.cpu().numpy())


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: neuralmind/bert-large-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🚀 Iniciando Fine-Tuning Avançado do BERTimbau Large...


Época 1/4:   0%|          | 0/2135 [00:00<?, ?it/s]

--> Época 1 | Acc: 0.4443 | F1-Macro: 0.4445
    [RECORD] Melhores pesos salvos em /content/drive/MyDrive/epPLN/ep2/bertimbau_large_ep2_best.pt


Época 2/4:   0%|          | 0/2135 [00:00<?, ?it/s]

--> Época 2 | Acc: 0.4628 | F1-Macro: 0.4612
    [RECORD] Melhores pesos salvos em /content/drive/MyDrive/epPLN/ep2/bertimbau_large_ep2_best.pt


Época 3/4:   0%|          | 0/2135 [00:00<?, ?it/s]

--> Época 3 | Acc: 0.4711 | F1-Macro: 0.4637
    [RECORD] Melhores pesos salvos em /content/drive/MyDrive/epPLN/ep2/bertimbau_large_ep2_best.pt


Época 4/4:   0%|          | 0/2135 [00:00<?, ?it/s]

--> Época 4 | Acc: 0.4725 | F1-Macro: 0.4652
    [RECORD] Melhores pesos salvos em /content/drive/MyDrive/epPLN/ep2/bertimbau_large_ep2_best.pt

GERANDO PREDIÇÕES OFICIAIS DO TESTE 2


In [ ]:
# Salva arquivo formatado para entrega do EP2
df_test2["clarity"] = [id2label[p] for p in preds_test2]
OUTPUT_TEST2_PATH = BASE_DIR / "grupoXX_teste2_rotulado.xlsx"
df_test2.to_excel(OUTPUT_TEST2_PATH, index=False)
print(f"✅ Planilha do Teste 2 pronta para entrega em: {OUTPUT_TEST2_PATH}")

✅ Planilha do Teste 2 pronta para entrega em: /content/drive/MyDrive/epPLN/ep2/grupoXX_teste2_rotulado.xlsx


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from statsmodels.stats.contingency_tables import mcnemar
from google.colab import drive

# 1. Montar o Google Drive (se necessário)
drive.mount("/content/drive")

# ==========================================
# 0. DIREITÓRIOS E CARREGAMENTO DOS ARQUIVOS
# ==========================================
BASE_DIR = Path("/content/drive/MyDrive/epPLN/ep2")

# Arquivo com as predições do EP1
EP1_PREDS_PATH = BASE_DIR / "test1_rotulado.xlsx"

# Arquivo oficial do professor com o GABARITO REAL
GABARITO_PATH = BASE_DIR / "test1-rotulado-para-teste-estatistico.xlsx"

label2id = {"c1": 0, "c234": 1, "c5": 2}

# Carrega os dataframes
df_ep1 = pd.read_excel(EP1_PREDS_PATH)
df_gabarito = pd.read_excel(GABARITO_PATH)

# 1. Predições do EP1 (pegas do arquivo test1_rotulado.xlsx)
preds_ep1 = df_ep1["clarity"].map(label2id).to_numpy()

# 2. Gabarito Real (pego do arquivo oficial do professor)
y_true = df_gabarito["clarity"].map(label2id).to_numpy()

# 3. Predições do EP2 (garanta que o vetor 'preds_ep2' do BERTimbau já foi gerado)
preds_ep2 = np.array(preds_ep2)

# ==========================================
# 1. VERIFICAÇÃO DE SEGURANÇA DOS TAMANHOS
# ==========================================
print(f"Tamanho Predições EP1: {len(preds_ep1)}")
print(f"Tamanho Predições EP2: {len(preds_ep2)}")
print(f"Tamanho Gabarito Real: {len(y_true)}")

if not (len(preds_ep1) == len(preds_ep2) == len(y_true)):
    raise ValueError(
        "❌ Os tamanhos dos vetores não coincidem! Verifique se todos os arquivos possuem 900 linhas."
    )

# ==========================================
# 2. CONSTRUÇÃO DA TABELA DE CONTINGÊNCIA (2x2)
# ==========================================
correct_ep1 = preds_ep1 == y_true
correct_ep2 = preds_ep2 == y_true

n11 = np.sum(correct_ep1 & correct_ep2)  # Ambos acertaram
n10 = np.sum(correct_ep1 & ~correct_ep2)  # EP1 acertou, EP2 errou
n01 = np.sum(~correct_ep1 & correct_ep2)  # EP1 errou, EP2 acertou
n00 = np.sum(~correct_ep1 & ~correct_ep2)  # Ambos erraram

table = [[n11, n10], [n01, n00]]

print("\n==========================================")
print("RESULTADO OFICIAL DO TESTE DE MCNEMAR (EP1 vs EP2)")
print("==========================================")
print(f"Tabela de Contingência (2x2):")
print(f"               EP2 Correto | EP2 Incorreto")
print(f"EP1 Correto   :    {n11:^7} |    {n10:^7}")
print(f"EP1 Incorreto :    {n01:^7} |    {n00:^7}\n")

# Accuracies para o relatório
acc_ep1 = np.mean(correct_ep1)
acc_ep2 = np.mean(correct_ep2)
print(f"Acurácia EP1: {acc_ep1:.4f}")
print(f"Acurácia EP2: {acc_ep2:.4f}\n")

# ==========================================
# 3. TESTE ESTATÍSTICO DE SIGNIFICÂNCIA
# ==========================================
discordantes = n10 + n01

if discordantes == 0:
    print(
        "⚠️ Os dois modelos obtiveram exatamente os mesmos acertos e erros."
    )
    print("p-valor: 1.000000 (Sem diferença estatística)")
else:
    mcnemar_result = mcnemar(table, exact=False, correction=True)
    print(f"Estatística qui-quadrado (χ²): {mcnemar_result.statistic:.4f}")
    print(f"p-valor: {mcnemar_result.pvalue:.6f}\n")

    if mcnemar_result.pvalue < 0.05:
        print(
            "✅ Conclusão: O EP2 é ESTATISTICAMENTE SUPERIOR ao EP1 (p < 0,05)!"
        )
    else:
        print(
            "⚠️ Conclusão: A diferença de desempenho entre os modelos não atingiu significância estatística (p >= 0,05)."
        )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Tamanho Predições EP1: 900
Tamanho Predições EP2: 900
Tamanho Gabarito Real: 900

RESULTADO OFICIAL DO TESTE DE MCNEMAR (EP1 vs EP2)
Tabela de Contingência (2x2):
               EP2 Correto | EP2 Incorreto
EP1 Correto   :      342   |      110  
EP1 Incorreto :      88    |      360  

Acurácia EP1: 0.5022
Acurácia EP2: 0.4778

Estatística qui-quadrado (χ²): 2.2273
p-valor: 0.135593

⚠️ Conclusão: A diferença de desempenho entre os modelos não atingiu significância estatística (p >= 0,05).


MELHORIA EP1: stacking/ensemble no bert escolhido + significância

Analise de erros

Analise de palavras com LIME

In [ ]:
!pip install lime

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283913 sha256=45b859810b0da6807de769079e522be0189a943f4ad7605e62c1f4988aa23287
  Stored in directory: /root/.cache/pip/wheels/7c/04/5c/157dc9106512a6c7a30653ec064490c94a49e0fc8f63d19ab9
Successfully built lime


In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from lime.lime_text import LimeTextExplainer
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)
from pathlib import Path
from google.colab import drive
from tqdm.notebook import tqdm

In [ ]:
# 1. Montar o Google Drive
drive.mount("/content/drive")

# ==========================================
# 0. AMBIENTE E SEED OFICIAL (123)
# ==========================================
SEED = 123
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_DIR = Path("/content/drive/MyDrive/epPLN/ep2/LIME_Rafa")
BASE_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = BASE_DIR / "train.xlsx"
CHECKPOINT_PATH = BASE_DIR / "bertimbau_large_best.pt"
REPORT_PATH = BASE_DIR / "bertimbau_large_lime.txt"

def write_report(text):
    with REPORT_PATH.open("a", encoding="utf-8") as report_file:
        report_file.write(f"{text}\n")

REPORT_PATH.write_text(
    "BERTIMBAU LARGE - MÉTRICAS E INTERPRETABILIDADE LIME (EXECUÇÃO SEM RETREINO)\n"
    "==========================================================================\n",
    encoding="utf-8",
)

print(f"Utilizando dispositivo: {device}")
print(f"Dataset: {DATA_PATH}")
print(f"Checkpoint do Modelo: {CHECKPOINT_PATH}")

df = pd.read_excel(DATA_PATH)
print("Dimensões do dataset:", df.shape)

def normalizar_texto(texto):
    if not isinstance(texto, str):
        return ""
    texto = re.sub(r"https?://\S+|www\.\S+", "LINK_URL", texto)
    texto = re.sub(
        r"(?i)\b(lei|decreto)\s*(?:nº|n°|nº\.|º|°)?\s*\d+[\d\./]*",
        "LEI_DECRETO",
        texto,
    )
    texto = re.sub(r"\b\d{5,}\.\d{6}/\d{4}-\d{2}\b", "NUMERO_PROCESSO", texto)
    texto = re.sub(
        r"\(\?\d{2,4}\)?\s*\d{4,5}[-\s]?\d{4}",
        "NUMERO_TELEFONE",
        texto,
    )
    return texto.lower()

df["resp_text_clean"] = df["resp_text"].apply(normalizar_texto)

label2id = {"c1": 0, "c234": 1, "c5": 2}
id2label = {value: key for key, value in label2id.items()}
df["label"] = df["clarity"].map(label2id)

_, df_val = train_test_split(
    df,
    test_size=0.20,
    random_state=SEED,
    stratify=df["label"],
)

# ==========================================
# 1. CARREGAMENTO DO MODELO SALVO
# ==========================================
MODEL_NAME = "neuralmind/bert-large-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LEN = 256

def encode_texts(texts):
    return tokenizer(
        [str(text) for text in texts],
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt",
    )

class TextDataset(Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = torch.tensor(labels, dtype=torch.long) if labels is not None else None

    def __getitem__(self, idx):
        item = {key: value[idx] for key, value in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.encodings["input_ids"])

val_dataset = TextDataset(
    encode_texts(df_val["resp_text_clean"].to_numpy()),
    df_val["label"].to_numpy(),
)

val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

print("\nInstanciando arquitetura e carregando pesos do arquivo .pt...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
).to(device)

model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
model.eval()
print("Modelo carregado com sucesso!")

# ==========================================
# 2. AVALIAÇÃO DO MODELO
# ==========================================
print("\n==================================================")
print("AVALIANDO DESEMPENHO NO CONJUNTO DE VALIDAÇÃO")
print("==================================================")

preds_all, labels_all = [], []
with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)
        preds_all.extend(preds.cpu().numpy())
        labels_all.extend(batch["labels"].numpy())

final_accuracy = accuracy_score(labels_all, preds_all)
final_f1_macro = f1_score(labels_all, preds_all, average="macro")
final_report = classification_report(
    labels_all,
    preds_all,
    target_names=["c1", "c234", "c5"],
    digits=4,
)

print(f"Val Accuracy: {final_accuracy:.4f}")
print(f"Val F1-Macro: {final_f1_macro:.4f}")
print(final_report)

write_report(f"Val Accuracy: {final_accuracy:.4f}")
write_report(f"Val F1-Macro: {final_f1_macro:.4f}")
write_report(final_report)

# ==========================================
# 3. INTERPRETABILIDADE LOCAL COM LIME
# ==========================================
LIME_NUM_EXAMPLES = 3
LIME_NUM_SAMPLES = 500
LIME_BATCH_SIZE = 16
LIME_NUM_FEATURES = 10

lime_explainer = LimeTextExplainer(
    class_names=[id2label[index] for index in range(len(id2label))],
    random_state=SEED,
)

def predict_proba(texts):
    model.eval()
    probabilities = []

    for start in range(0, len(texts), LIME_BATCH_SIZE):
        batch_texts = [
            normalizar_texto(text)
            for text in texts[start : start + LIME_BATCH_SIZE]
        ]
        encodings = tokenizer(
            batch_texts,
            padding="max_length",
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt",
        )
        encodings = {key: value.to(device) for key, value in encodings.items()}

        with torch.no_grad():
            logits = model(**encodings).logits
            batch_probabilities = torch.softmax(logits, dim=1)
        probabilities.extend(batch_probabilities.cpu().numpy())

    return np.asarray(probabilities)

print(f"\nGerando explicações LIME para os primeiros {LIME_NUM_EXAMPLES} exemplos...")

for example_number, (_, row) in enumerate(
    df_val.head(LIME_NUM_EXAMPLES).iterrows(),
    start=1,
):
    text = row["resp_text_clean"]
    true_label_id = int(row["label"])
    probabilities = predict_proba([text])
    predicted_label_id = int(np.argmax(probabilities[0]))

    explanation = lime_explainer.explain_instance(
        text,
        predict_proba,
        labels=tuple(range(len(id2label))),
        num_features=LIME_NUM_FEATURES,
        num_samples=LIME_NUM_SAMPLES,
    )

    probability_text = ", ".join(
        f"{id2label[index]}: {probability:.4f}"
        for index, probability in enumerate(probabilities[0])
    )
    sample_report = (
        f"\nExemplo {example_number}\n"
        f"Rótulo real: {id2label[true_label_id]}\n"
        f"Rótulo previsto: {id2label[predicted_label_id]}\n"
        f"Probabilidades: {probability_text}\n"
        f"Texto analisado: {text}\n"
    )

    for class_id, class_name in id2label.items():
        sample_report += (
            f"Contribuições LIME para {class_name}: "
            f"{explanation.as_list(label=class_id)}\n"
        )

    print(sample_report)
    write_report(sample_report)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Utilizando dispositivo: cuda
Dataset: /content/drive/MyDrive/epPLN/ep2/LIME_Rafa/train.xlsx
Checkpoint do Modelo: /content/drive/MyDrive/epPLN/ep2/LIME_Rafa/bertimbau_large_best.pt
Dimensões do dataset: (20092, 2)

Instanciando arquitetura e carregando pesos do arquivo .pt...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-large-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from t

Modelo carregado com sucesso!

AVALIANDO DESEMPENHO NO CONJUNTO DE VALIDAÇÃO
Val Accuracy: 0.4710
Val F1-Macro: 0.4612
              precision    recall  f1-score   support

          c1     0.4674    0.5713    0.5142      1269
        c234     0.4116    0.2750    0.3297      1371
          c5     0.5097    0.5736    0.5397      1379

    accuracy                         0.4710      4019
   macro avg     0.4629    0.4733    0.4612      4019
weighted avg     0.4629    0.4710    0.4600      4019


Gerando explicações LIME para os primeiros 3 exemplos...

Exemplo 1
Rótulo real: c5
Rótulo previsto: c1
Probabilidades: c1: 0.6487, c234: 0.2031, c5: 0.1483
Texto analisado:  prezado,  segue resposta ao seu pedido de informação, conforme documento anexo. em caso de insatisfação, existe possibilidade de ingressar com recurso no sistema.  att.  sic ifba 
Contribuições LIME para c1: [(np.str_('prezado'), -0.08347776682912852), (np.str_('ifba'), -0.060027507287366985), (np.str_('insatisfação'), 0.0

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from lime.lime_text import LimeTextExplainer
import torch

# ==========================================
# DEFINIÇÃO DO ARQUIVO DE SAÍDA
# ==========================================
OUTPUT_TXT_PATH = BASE_DIR / "saida-baseline.txt"

# Inicializa/Sobrescreve o arquivo com o cabeçalho
OUTPUT_TXT_PATH.write_text(
    "PESOS DAS PALAVRAS E IMPORTÂNCIA DE FEATURES (LIME)\n"
    "===================================================\n\n",
    encoding="utf-8",
)


def exportar_pesos_lime(text_sample, sample_id, true_class, pred_class):
    explanation = lime_explainer.explain_instance(
        text_sample,
        predict_proba,
        labels=tuple(range(len(id2label))),
        num_features=15,  # Quantidade de palavras/features por exemplo
        num_samples=500,
    )

    linhas = []
    linhas.append(f"--- EXEMPLO #{sample_id} ---")
    linhas.append(f"Texto: {text_sample}")
    linhas.append(f"Classe Real: {true_class} | Classe Prevista: {pred_class}\n")
    linhas.append("PESOS DAS PALAVRAS POR CLASSE:")

    # Percorre cada classe e extrai os pares (palavra, peso)
    for class_id, class_name in id2label.items():
        linhas.append(f"\n  [Classe: {class_name}]")
        pesos = explanation.as_list(label=class_id)

        for palavra, peso in pesos:
            # Indica se a palavra puxa A FAVOR (+) ou CONTRA (-) a classe
            sinal = "+" if peso > 0 else ""
            linhas.append(f"    - {palavra:<20} : {sinal}{peso:.6f}")

    linhas.append("\n" + "=" * 50 + "\n")

    # Escreve a amostra no arquivo TXT
    with OUTPUT_TXT_PATH.open("a", encoding="utf-8") as f:
        f.write("\n".join(linhas) + "\n")


# ==========================================
# PROCESSAMENTO DE EXEMPLOS DA VALIDAÇÃO
# ==========================================
# Altere o número de exemplos conforme a necessidade (ex: 10, 20 ou len(df_val))
NUM_EXEMPLOS_PARA_EXPORTAR = 5

print(
    f"Gerando 'saida-baseline.txt' para {NUM_EXEMPLOS_PARA_EXPORTAR} exemplos..."
)

for idx, (_, row) in enumerate(
    df_val.head(NUM_EXEMPLOS_PARA_EXPORTAR).iterrows(), start=1
):
    texto = row["resp_text_clean"]
    rotulo_real = id2label[int(row["label"])]

    # Predição do modelo
    probs = predict_proba([texto])[0]
    rotulo_predito = id2label[int(np.argmax(probs))]

    # Gera e salva a explicação do LIME
    exportar_pesos_lime(
        texto, sample_id=idx, true_class=rotulo_real, pred_class=rotulo_predito
    )

print(f"✅ Arquivo salvo com sucesso em: {OUTPUT_TXT_PATH}")

Gerando 'saida-baseline.txt' para 5 exemplos...
✅ Arquivo salvo com sucesso em: /content/drive/MyDrive/epPLN/ep2/LIME_Rafa/saida-baseline.txt
